# 06 - OCR Motoru Karşılaştırması

PaddleOCR vs EasyOCR Türk plaka okuma performansı.

## Metrikler
- Doğru okuma oranı (tam eşleşme)
- Karakter düzeyinde doğruluk
- Geçerli Türk plaka formatı oranı
- Ortalama işlem süresi

In [ ]:
# !pip install paddlepaddle paddleocr easyocr

import sys
sys.path.insert(0, '..')

import cv2
import numpy as np
import matplotlib.pyplot as plt
import time
from pathlib import Path
import pandas as pd

from src.alpr.plate_reader import PlateReader
from src.alpr.plate_preprocessor import PlatePreprocessor
from src.alpr.plate_validator import PlateValidator

In [ ]:
# Test plaka görüntüleri
PLATE_DIR = Path('data/datasets/plate_detection/images/test')

# Ground truth (manuel oluşturulmalı)
# Format: {dosya_adı: gerçek_plaka_metni}
GROUND_TRUTH = {
    # Örnek:
    # 'plate_001.jpg': '34ABC1234',
    # 'plate_002.jpg': '06DE567',
}

In [ ]:
def evaluate_ocr_engine(engine_name, plate_images, ground_truth):
    """OCR motorunu değerlendir."""
    reader = PlateReader(engine=engine_name)
    validator = PlateValidator()
    
    results = []
    for img_path, gt_text in ground_truth.items():
        img = cv2.imread(str(plate_images / img_path))
        if img is None:
            continue
        
        t0 = time.time()
        result = reader.read_with_retry(img)
        elapsed = time.time() - t0
        
        exact_match = result.plate_text == gt_text.replace(' ', '')
        
        results.append({
            'file': img_path,
            'ground_truth': gt_text,
            'predicted': result.plate_text,
            'confidence': result.confidence,
            'is_valid': result.is_valid,
            'exact_match': exact_match,
            'time_ms': elapsed * 1000,
        })
    
    return pd.DataFrame(results)

if GROUND_TRUTH:
    print('PaddleOCR değerlendiriliyor...')
    df_paddle = evaluate_ocr_engine('paddleocr', PLATE_DIR, GROUND_TRUTH)
    
    print('EasyOCR değerlendiriliyor...')
    df_easy = evaluate_ocr_engine('easyocr', PLATE_DIR, GROUND_TRUTH)
    
    # Özet
    summary = pd.DataFrame({
        'Metrik': ['Doğru Okuma Oranı', 'Geçerli Plaka Oranı', 'Ort. Güvenilirlik', 'Ort. Süre (ms)'],
        'PaddleOCR': [
            f"{df_paddle['exact_match'].mean():.1%}",
            f"{df_paddle['is_valid'].mean():.1%}",
            f"{df_paddle['confidence'].mean():.3f}",
            f"{df_paddle['time_ms'].mean():.1f}",
        ],
        'EasyOCR': [
            f"{df_easy['exact_match'].mean():.1%}",
            f"{df_easy['is_valid'].mean():.1%}",
            f"{df_easy['confidence'].mean():.3f}",
            f"{df_easy['time_ms'].mean():.1f}",
        ],
    })
    print('\n', summary.to_string(index=False))
    summary.to_csv('results/ocr_comparison.csv', index=False)
else:
    print('Ground truth verisi girilmedi. GROUND_TRUTH sözlüğünü doldurun.')